In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All libraries impoerted successfully")
print("Ready to build a RAG system.")

All libraries impoerted successfully
Ready to build a RAG system.


In [ ]:
GROQ_API_KEY ="groq_api_key_here" # IGNORE --- REPLACE WITH YOUR GROQ API KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized.")
print("Note: If you see an aythentication error later, double-check your GROQ_API_KEY")

Groq API client initialized.
Note: If you see an aythentication error later, double-check your GROQ_API_KEY


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/codebooster-datasets/college_notes.csv')

print("Shape of dataset:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
   note_id           subject             topic  \
0        1  Data Engineering     ETL Pipelines   
1        2  Data Engineering  Data Warehousing   
2        3  Data Engineering      Apache Spark   

                                             content  
0  ETL stands for Extract, Transform, Load. It is...  
1  A data warehouse is a central repository that ...  
2  Apache Spark is an open-source distributed com...  


In [ ]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\nSample of topics:")
print(df[['note_id', 'subject', 'topic']].to_string(index=False))
print("\nLength of content (number od of characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Subjects in the dataset:
subject
Data Engineering    5
GenAI               5
Machine Learning    3
Python              2
Name: count, dtype: int64

Sample of topics:
 note_id          subject                  topic
       1 Data Engineering          ETL Pipelines
       2 Data Engineering       Data Warehousing
       3 Data Engineering           Apache Spark
       4 Data Engineering Medallion Architecture
       5 Data Engineering         Data Pipelines
       6 Machine Learning      Linear Regression
       7 Machine Learning    Feature Engineering
       8 Machine Learning       Model Evaluation
       9            GenAI  Large Language Models
      10            GenAI     Prompt Engineering
      11            GenAI             Embeddings
      12            GenAI       Vector Databases
      13            GenAI            RAG Systems
      14           Python         Pandas Library
      15           Python        API Integration

Length of content (number od of characters) for e

In [ ]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First documents ID: {ids[0]}")
print(f"First metadata: {metadata[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared: 15
First documents ID: note_1
First metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from ...


In [ ]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First documents ID: {ids[14]}")
print(f"First metadata: {metadata[14]}")
print(f"First 100 chars of doc: {documents[14][:100]}...")

Total chunks prepared: 15
First documents ID: note_15
First metadata: {'subject': 'Python', 'topic': 'API Integration'}
First 100 chars of doc: An API or Application Programming Interface allows different software systems to communicate with ea...


In [ ]:
print("Loding embedding model")
print("(Subsequent run will be faster as the model is cached)")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfuly.")
test_embedding = embedding_model.encode("This is a test sentence")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of test embedding: {test_embedding[:5]}")

Loding embedding model
(Subsequent run will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfuly.
Test embedding shape: (384,)
First 5 values of test embedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")

print("ChromaDB client created.")
print(f"Collection name: college_notes_rag")
print(f"Document in collection so far: {collection.count()}")

ChromaDB client created.
Collection name: college_notes_rag
Document in collection so far: 0


In [ ]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds")
embedding = embedding_model.encode(documents, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {embedding.shape}")
embedding_list = embedding.tolist()

collection.add(
    documents=documents,
    embeddings=embedding_list,
    ids=ids,
    metadatas=metadata
)

print(f"\nDocuments successfully added to chromaDB")
print(f"Total documents in collection: {collection.count()}")

Generating embeddings for all 15 notes...
This may take 15-30 seconds


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)

Documents successfully added to chromaDB
Total documents in collection: 15


In [ ]:
def retrieve_relevent_chunks(question, top_k=3):
    """
    Give datas are the collection of subject details.
    A dictionary containing retrived documents.
    """

    question_embedding = embedding_model.encode(question).tolist()

    result = collection.query(
        query_embeddings = [question_embedding],
        n_results = top_k
    )

    return result
print("Function to retrieve relevent chunks is created.")

Function to retrieve relevent chunks is created.


In [ ]:
test_question = "What is ETL and how does it work in data engineering ?"
print(f"Test QUestion: {test_question}")
print("="*60)

result = retrieve_relevent_chunks(test_question, top_k = 3)

print("\n Top 3 Retrieved Chunks:")
print("="* 60)
for i, (doc, dist, meta) in enumerate(zip(
    result['documents'][0],
    result['distances'][0],
    result['metadatas'][0]
)):
   print(f"\nResult {i+1}:")
   print(f" Subject : {meta['subject']}")
   print(f" Topic : {meta['topic']}")
   print(f" Distance : {dist:.4f}")
   print(f" Chunk : {doc[:100]}...")

Test QUestion: What is ETL and how does it work in data engineering ?

 Top 3 Retrieved Chunks:

Result 1:
 Subject : Data Engineering
 Topic : ETL Pipelines
 Distance : 0.3070
 Chunk : ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from ...

Result 2:
 Subject : Data Engineering
 Topic : Data Pipelines
 Distance : 1.2881
 Chunk : A data pipeline is a series of automated steps that move and transform data from one system to anoth...

Result 3:
 Subject : Data Engineering
 Topic : Medallion Architecture
 Distance : 1.3115
 Chunk : Medallion Architecture is a data design pattern used in modern data lakes and lakehouses. It organiz...


In [ ]:
test_question = "What is GenAI and how it works ?"
print(f"Test QUestion: {test_question}")
print("="*60)

result = retrieve_relevent_chunks(test_question, top_k = 3)

print("\n Top 3 Retrieved Chunks:")
print("="* 60)
for i, (doc, dist, meta) in enumerate(zip(
    result['documents'][0],
    result['distances'][0],
    result['metadatas'][0]
)):
   print(f"\nResult {i+1}:")
   print(f" Subject : {meta['subject']}")
   print(f" Topic : {meta['topic']}")
   print(f" Distance : {dist:.4f}")
   print(f" Chunk : {doc[:100]}...")

Test QUestion: What is GenAI and how it works ?

 Top 3 Retrieved Chunks:

Result 1:
 Subject : GenAI
 Topic : RAG Systems
 Distance : 1.4325
 Chunk : Retrieval-Augmented Generation or RAG is an AI architecture that improves LLM responses by retrievin...

Result 2:
 Subject : GenAI
 Topic : Large Language Models
 Distance : 1.4603
 Chunk : A Large Language Model or LLM is a type of artificial intelligence trained on massive amounts of tex...

Result 3:
 Subject : Machine Learning
 Topic : Feature Engineering
 Distance : 1.5975
 Chunk : Feature engineering is the process of selecting, creating, and transforming raw data variables calle...


In [ ]:
def build_context_from_results(results):
  """
  Format ChromaDB retrieval res+ults into a readable context string.
  """

  context_parts = []

  for i, (doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text=f"[source{i+1}:{meta['subject']} - {meta['topic']}]\n{doc}"

    context_parts.append(chunk_text)

  context_str="\n\n---\n\n".join(context_parts)

  return context_str
context = build_context_from_results(result)
print("Built context string from retrieved chunks:")
print("="*60)
print(context[:500] + "...")
print(f"\nTotal context length:{len(context)} characters")

Built context string from retrieved chunks:
[source1:GenAI - RAG Systems]
Retrieval-Augmented Generation or RAG is an AI architecture that improves LLM responses by retrieving relevant documents from a knowledge base and injecting them into the prompt as context. This solves the hallucination problem where LLMs generate incorrect information. The RAG pipeline works in steps: first documents are chunked and embedded into vectors, then stored in a vector database, then when a user asks a question the question is embedded and used to retriev...

Total context length:1594 characters


In [ ]:
def generate_rag_answer(question, context):
    system_prompt = """You are a helpful academic assistant for engineering students.


    You will be given context retrieved from a college knowledge base, and a student's question.


    RULES:
    1. Answer ONLY using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible."""


    # USER PROMPT: The context + question formatted as the user message
    user_prompt = f"""Context from Knowledge Base:


{context}


---


Student's Question: {question}


Please answer the question based only on the context provided above."""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )
    answer = response.choices[0].message.content

    return answer

print("RAG generation function defined.")

RAG generation function defined.


#RAG Pipeline
question -> retrieve -> inject -> generate ->answer

In [ ]:
def ask_college_assistant(question,top_k=3, verbose =True):
  if verbose:
    print(f"Question: {question}")
    print("="*60)
    print("Step 1: Retrieving relevant documents...")

  results = retrieve_relevent_chunks(question, top_k=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base.")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f"{i+1}.{meta['subject']} - {meta['topic']}")
    print("\nStep 2: Building context string...")

  context = build_context_from_results(results)

  if verbose:
    print(f"Context built ({len(context)} characters)")
    print("\nStep 3: Sending to LLM for answer generation...")

  answer=generate_rag_answer(question, context)

  if verbose:
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("=" * 60)
    print(answer)

  return answer

print("Complete RAG pipeline function ready.")
print("Function : ask_college_assistant(question, top_k=3)")

Complete RAG pipeline function ready.
Function : ask_college_assistant(question, top_k=3)


In [ ]:
GROQ_API_KEY ="groq_api_key_here" # IGNORE --- REPLACE WITH YOUR GROQ API KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized.")
print("Note: If you see an aythentication error later, double-check your GROQ_API_KEY")

Groq API client initialized.
Note: If you see an aythentication error later, double-check your GROQ_API_KEY


In [ ]:
question1="What is ETL and what are its three main stages?"

answer1=ask_college_assistant(question1,top_k=3,verbose=True)

print()

question2="Who is the founder of meta?"

answer2=ask_college_assistant(question2,top_k=3,verbose=True)

print()

question3="What is GenAI and how it works ?"

answer3=ask_college_assistant(question3,top_k=3,verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base.
1.Data Engineering - ETL Pipelines
2.GenAI - Prompt Engineering
3.Data Engineering - Medallion Architecture

Step 2: Building context string...
Context built (1584 characters)

Step 3: Sending to LLM for answer generation...

ANSWER:
ETL stands for Extract, Transform, Load. 

Its three main stages are:

1. **Extract**: This is the first stage where data is extracted from multiple sources such as databases, APIs, or files.
2. **Transform**: In this stage, the extracted data is cleaned, errors are corrected, formats are converted, and business rules are applied.
3. **Load**: The final stage where the cleaned data is loaded into a destination system such as a data warehouse or data lake.

This information comes from [source1: Data Engineering - ETL Pipelines].

Question: Who is the founder of meta?
Step 1: Retrieving relevant documents...
Retrieved